# # Problem Set 1 — Markov decision process and dynamic programming

**Goal.** Practice Dynamic programming and deal with both pen and paper problem and coding on gym *cliffwalk* environment.

**TODO list**
- 111

## 1. Inscribed Polygon of Maximal Perimeter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----- Grid parameters -----
H, W = 4, 12  # height, width
N = H * W     # number of states (each grid cell is a state)
ACTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # up, right, down, left
A_NAMES = ["↑", "→", "↓", "←"]
gamma = 0.9

START = (3, 0)
GOAL  = (3, 11)
CLIFF = [(3, j) for j in range(1, W-1)]  # cliff cells along the bottom row except start/goal

def idx(i, j):
    return i * W + j

def in_bounds(i, j):
    return 0 <= i < H and 0 <= j < W

# ----- Build P^a and r^a -----
nA = len(ACTIONS)
P = np.zeros((nA, N, N))   # transition matrices
r = np.zeros((nA, N))      # reward vectors

S_idx = idx(*START)
G_idx = idx(*GOAL)

for i in range(H):
    for j in range(W):
        s = idx(i, j)

        # Goal is absorbing with 0 reward
        if (i, j) == GOAL:
            for a in range(nA):
                P[a, s, s] = 1.0
                r[a, s] = 0.0
            continue

        # For all other states (including START)
        for a, (di, dj) in enumerate(ACTIONS):
            ni, nj = i + di, j + dj
            if not in_bounds(ni, nj):
                # Hitting wall: stay in place
                ni, nj = i, j

            # If next cell is cliff: teleport to START with -100 reward
            if (ni, nj) in CLIFF:
                ns = S_idx
                P[a, s, ns] = 1.0
                r[a, s] = -100.0
                continue

            # If next is GOAL: absorb with 0 immediate reward
            if (ni, nj) == GOAL:
                ns = G_idx
                P[a, s, ns] = 1.0
                r[a, s] = 0.0  # alternative: -1
                continue

            # Regular move: -1 reward
            ns = idx(ni, nj)
            P[a, s, ns] = 1.0
            r[a, s] = -1.0

print("Transition and reward tensors built:")
print("P shape:", P.shape, "r shape:", r.shape)


In [ ]:
# ----- Value Iteration in matrix–vector form -----
def value_iteration(P, r, gamma=0.9, max_iter=1000, tol=1e-8):
    nA, N, _ = P.shape
    V = np.zeros(N)
    for k in range(max_iter):
        Q = np.zeros((nA, N))
        for a in range(nA):
            Q[a] = r[a] + gamma * P[a].dot(V)
        V_new = np.max(Q, axis=0)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, k+1
        V = V_new
    return V, max_iter

V_opt, iters = value_iteration(P, r, gamma=gamma, max_iter=5000, tol=1e-10)
print(f"Converged in {iters} iterations.")
V_grid = V_opt.reshape(H, W)
V_grid


In [ ]:
# ----- Extract greedy policy from V -----
def greedy_policy(P, r, V, gamma=0.9):
    nA, N, _ = P.shape
    Q = np.zeros((nA, N))
    for a in range(nA):
        Q[a] = r[a] + gamma * P[a].dot(V)
    pi = np.argmax(Q, axis=0)
    return pi, Q

pi_opt, Q_opt = greedy_policy(P, r, V_opt, gamma)
pi_grid = pi_opt.reshape(H, W)
pi_grid


In [ ]:
# ----- Visualize value function and greedy policy -----
fig, ax = plt.subplots(figsize=(10, 3.5))
im = ax.imshow(V_grid, origin='upper')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='V*')

for (ci, cj) in CLIFF:
    ax.text(cj, ci, 'X', ha='center', va='center')
ax.text(GOAL[1], GOAL[0], 'G', ha='center', va='center')
ax.text(START[1], START[0], 'S', ha='center', va='center')

ax.set_title("Cliffwalk: Optimal Value Function V* (matrix–vector DP)")
ax.set_xticks(range(W)); ax.set_yticks(range(H))
ax.set_xticklabels(range(W)); ax.set_yticklabels(range(H))
ax.grid(which='major', color='k', linewidth=0.5)

arrow = {0:(-0.35,0), 1:(0,0.35), 2:(0.35,0), 3:(0,-0.35)}
for i in range(H):
    for j in range(W):
        a = pi_grid[i, j]
        dy, dx = arrow[a]
        ax.arrow(j, i, dx, dy, head_width=0.2, head_length=0.15, length_includes_head=True)

plt.tight_layout()
plt.show()


### Notes
- Deterministic Cliffwalk; to add stochasticity, spread each row of $P^a$ across neighbors.
- Goal is absorbing with $0$ reward; you can set entering reward to $-1$ if preferred.
- Bellman update uses vectorized matrix–vector products to update all states per sweep.
